# EasyRAG Manual Document Preparation

## What you will do

- create custom documents with explicit IDs and logical paths
- preview the metadata before insertion
- insert prepared `Document` objects with `ainsert_documents()`
- compare that path with one direct `ainsert()` call
- verify that citations preserve the logical titles and paths you chose

## Why this matters

Repository loading is only one entry path. Evaluation sets, scratch notes, and application-specific corpora often begin as in-memory texts, so the manual path needs to feel just as first-class as the repo loader.

## Step 1: Import the public API and runtime helper

We only need the high-level `EasyRAG` API, `QueryParam`, and `run_sync` for this notebook. The rest of the behavior will come from deterministic stubs and prepared documents.


In [ ]:
import json
import tempfile

from easyrag import EasyRAG, QueryParam
from easyrag.support.async_utils import run_sync


This cell should only import symbols. The interesting part starts once we define the helpers and custom texts.


## Step 2: Define deterministic embedding and query-model stubs

These stubs keep the notebook zero-key runnable. The goal is not to simulate a perfect model. The goal is to make insertion and retrieval behavior stable enough for teaching.


In [ ]:
_KEYWORDS = [
    "faq",
    "api",
    "guide",
    "pricing",
    "workflow",
    "retrieval",
    "query",
    "support",
    "policy",
    "document",
    "insert",
]


def _stub_embedding(texts: list[str]) -> list[list[float]]:
    vectors = []
    for text in texts:
        lowered = text.lower()
        vector = [float(lowered.count(keyword)) for keyword in _KEYWORDS]
        vector.append(float(len(lowered.split())))
        vectors.append(vector)
    return vectors


def _stub_query_model(prompt: str, *, task: str, count: int = 1) -> str | list[str]:
    cleaned = prompt.split(":", 1)[-1].strip()
    if task == "rewrite":
        return cleaned
    if task == "mqe":
        return [f"{cleaned} variant {index}" for index in range(1, count + 1)]
    raise ValueError(task)


This cell defines the deterministic helpers only. There should be no runtime output yet.


## Step 3: Prepare custom `Document` objects with explicit metadata

`EasyRAG.prepare_documents()` is the easiest way to build canonical `Document` objects from in-memory text. The key idea is that you control the document IDs and logical file paths yourself. Those choices later affect ownership, upsert behavior, and citation output.


In [ ]:
prepared_documents = EasyRAG.prepare_documents(
    [
        "# FAQ\nEasyRAG supports local-first retrieval for documentation teams.\n",
        "# API Guide\nUse `ainsert` for raw texts and `ainsert_documents` for prepared `Document` objects.\n",
    ],
    ids=["doc::faq", "doc::api-guide"],
    file_paths=["handbook/faq.md", "handbook/api-guide.md"],
)

preview = [
    {
        "doc_id": document.metadata["doc_id"],
        "title": document.metadata["title"],
        "relative_path": document.metadata["relative_path"],
        "source_type": document.metadata["source_type"],
    }
    for document in prepared_documents
]

print(json.dumps(preview, indent=2))


You should see the exact IDs and logical file paths you supplied. This is the clearest proof that manual ingestion can preserve application-defined metadata instead of only filesystem-derived metadata.


## Step 4: Create and initialize a temporary workspace

We build the custom-documents example inside a temporary directory so the notebook remains self-contained and repeatable.


In [ ]:
temp_dir = tempfile.TemporaryDirectory()
working_dir = temp_dir.name
workspace = "custom-documents"

rag = EasyRAG(
    working_dir=working_dir,
    workspace=workspace,
    embedding_func=_stub_embedding,
    query_model_func=_stub_query_model,
)

run_sync(rag.initialize_storages())
print(json.dumps({"working_dir": working_dir, "workspace": workspace}, indent=2))


The workspace now exists, but it does not contain any indexed knowledge yet. The next cell inserts the prepared documents you just inspected.


## Step 5: Insert the prepared documents

`ainsert_documents()` is the right entry point when you already have canonical `Document` objects and want to preserve their metadata exactly as prepared.


In [ ]:
prepared_insert_stats = run_sync(rag.ainsert_documents(prepared_documents))
print(json.dumps(prepared_insert_stats, indent=2))


A successful output should show nonzero document and chunk counts. At this point, the workspace contains retrievable records built from manually prepared metadata.


## Step 6: Add one direct raw-text insert for contrast

`ainsert()` is the simpler path when you do not already have `Document` objects. It still creates canonical metadata, but the normalization happens inside the orchestrator rather than before the call.


In [ ]:
raw_insert_stats = run_sync(
    rag.ainsert(
        "# Pricing Notes\nThe pricing workflow explains how support plans map to retrieval quotas.\n",
        ids=["doc::pricing-notes"],
        file_paths=["notes/pricing-notes.md"],
    )
)

print(json.dumps(raw_insert_stats, indent=2))


This second insert gives the workspace one more manually defined record. The important contrast is not the stats themselves. The important contrast is where the metadata normalization happened: before the call in one case, inside the call in the other.


## Step 7: Query the workspace and inspect grounded citations

Now that the custom documents are indexed, we can verify that the logical paths and titles survive all the way into retrieval output.


In [ ]:
result = run_sync(
    rag.aquery(
        "How do I insert prepared documents?",
        QueryParam(mode="naive", rewrite_enabled=False, mqe_enabled=False),
    )
)

print(json.dumps({
    "citations": result.citations,
    "metadata": result.metadata,
}, indent=2))


You should see citation payloads that point to logical paths such as `handbook/api-guide.md`. That is the main takeaway of the notebook: manually chosen metadata is not lost during insertion or retrieval.


## Step 8: Inspect workspace stats after both insertion paths

It is useful to ask the workspace how much indexed state now exists. This gives you a compact summary of what the prepared-document path and the raw-text path created together.


In [ ]:
stats = run_sync(rag.get_stats())
print(json.dumps(stats, indent=2))


This output should confirm that the workspace holds the combined result of both insertion styles. It is one workspace, even though the knowledge entered through two different API paths.


## Step 9: Clean up the workspace

We close the storages and remove the temporary directory so the notebook leaves no persistent state behind.


In [ ]:
run_sync(rag.finalize_storages())
temp_dir.cleanup()

print("Closed storages and removed the temporary custom-documents workspace.")


After cleanup, the manual-ingestion walkthrough is complete. The important lesson is that EasyRAG can index knowledge from more than one entry path while still preserving stable retrieval metadata.

## Next steps

- read [docs/03-indexing-overview.md](../../docs/03-indexing-overview.md) for the broader indexing picture
- compare this notebook with [03_07_build_index_pipeline.ipynb](../03_indexing/03_07_build_index_pipeline.ipynb) to contrast repo loading and manual ingestion
- revisit [00_01_quickstart_end_to_end.ipynb](../00_overview/00_01_quickstart_end_to_end.ipynb) if you want the smallest end-to-end loop again